# Notebook 04 - SFT With Adaption-Improved Data

This is the adapted-data SFT condition. It trains the same base model with the same training settings as Notebook 03, but uses `data/processed/kakugo_adapted_sft.jsonl`.

Keeping these notebooks parallel makes the comparison easier to reason about.

**Files read:**
- [`../configs/tinker_sft_adapted.yaml`](../configs/tinker_sft_adapted.yaml) - base model, renderer, training hyperparameters, input path, and output path for the adapted-data SFT run.
- [`../data/processed/kakugo_adapted_sft.jsonl`](../data/processed/kakugo_adapted_sft.jsonl) - chat-format adapted SFT examples prepared in Notebook 02.

**Files written:**
- [`../results/models/sft_adapted/`](../results/models/sft_adapted/) - Tinker logs, metrics, checkpoints, and final model/sampler references for the adapted-data SFT run.

In [1]:
import asyncio
import json
import logging
import os
import sys
import warnings
from pathlib import Path

import nest_asyncio
from huggingface_hub import login
from tinker_cookbook.supervised.data import FromConversationFileBuilder
from tinker_cookbook.supervised.train import Config, main
from tinker_cookbook.supervised.types import ChatDatasetBuilderCommonConfig

nest_asyncio.apply()
warnings.filterwarnings('ignore', module='tinker_cookbook')

In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs' / 'project.yaml').exists():
            return candidate
    raise FileNotFoundError(
        'Run this notebook from inside the adaption-kirundi-sft-starter repo.'
    )


ROOT = find_repo_root(Path.cwd())
SRC = str(ROOT / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [3]:
from kirundi_sft_starter.utils import ensure_dir, load_env, load_yaml, require_file

load_env()
adapted_sft_config = load_yaml(ROOT / 'configs/tinker_sft_adapted.yaml')

require_file(ROOT / adapted_sft_config['data_path'], 'Run Notebook 02 before launching adapted-data SFT.')
ensure_dir(ROOT / adapted_sft_config['output_dir'])

if not os.environ.get('TINKER_API_KEY'):
    raise RuntimeError('Missing TINKER_TOKEN or TINKER_API_KEY. Add it to .env before launching training.')

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Review the training config

This is the exact adapted-data SFT configuration that will be sent to Tinker in the training cell below.

In [4]:
print(json.dumps(adapted_sft_config, indent=2))

{
  "run_name": "sft_adapted",
  "base_model": "Qwen/Qwen3-8B-Base",
  "renderer_name": "role_colon",
  "data_path": "data/processed/kakugo_adapted_sft.jsonl",
  "output_dir": "results/models/sft_adapted",
  "learning_rate": "2e-4",
  "num_epochs": 3,
  "batch_size": 4,
  "lora_rank": 16,
  "max_length": 2048,
  "train_on_what": "last_assistant_message",
  "save_every": 50
}


## Build the Tinker SFT config

This mirrors Notebook 03. The only intentional difference is the data path and output directory from `configs/tinker_sft_adapted.yaml`.

In [5]:
renderer_name = adapted_sft_config['renderer_name']

common_config = ChatDatasetBuilderCommonConfig(
    model_name_for_tokenizer=adapted_sft_config['base_model'],
    renderer_name=renderer_name,
    max_length=int(adapted_sft_config['max_length']),
    batch_size=int(adapted_sft_config['batch_size']),
    train_on_what=adapted_sft_config['train_on_what'],
)

dataset_builder = FromConversationFileBuilder(
    file_path=str(ROOT / adapted_sft_config['data_path']),
    common_config=common_config,
)

sft_config = Config(
    log_path=str(ROOT / adapted_sft_config['output_dir']),
    model_name=adapted_sft_config['base_model'],
    dataset_builder=dataset_builder,
    learning_rate=float(adapted_sft_config['learning_rate']),
    lora_rank=int(adapted_sft_config['lora_rank']),
    num_epochs=int(adapted_sft_config['num_epochs']),
)

print('\nSFT Config:')
print(f"  Run:           {adapted_sft_config['run_name']}")
print(f"  Model:         {sft_config.model_name}")
print(f"  Renderer:      {renderer_name}")
print(f"  Data:          {ROOT / adapted_sft_config['data_path']}")
print(f"  Output:        {sft_config.log_path}")
print(f"  Learning rate: {sft_config.learning_rate}")
print(f"  LoRA rank:     {sft_config.lora_rank}")
print(f"  Epochs:        {sft_config.num_epochs}")


SFT Config:
  Run:           sft_adapted
  Model:         Qwen/Qwen3-8B-Base
  Renderer:      role_colon
  Data:          /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/data/processed/kakugo_adapted_sft.jsonl
  Output:        /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/models/sft_adapted
  Learning rate: 0.0002
  LoRA rank:     16
  Epochs:        3


## Launch training

This cell starts the real Tinker SFT job. If credentials, package imports, data paths, or Tinker access are wrong, the cell should fail loudly so the issue is visible.

In [6]:
print('=' * 50)
print('Starting adapted-data SFT training...')
print('Watch train_mean_nll; it should generally decrease across training.\n')

logging.getLogger('asyncio').setLevel(logging.CRITICAL)
result = asyncio.get_event_loop().run_until_complete(main(sft_config))

print('\nTraining complete!')
print(f'Result: {result}')

Starting adapted-data SFT training...
Watch train_mean_nll; it should generally decrease across training.



root:680 [INFO] Command line invocation: /Users/vivianamarquez/anaconda3/envs/adaption-kirundi-sft/lib/python3.11/site-packages/ipykernel_launcher.py -f /Users/vivianamarquez/Library/Jupyter/runtime/kernel-186ec471-9ac1-4d29-8d0b-882aa18d543d.json
tinker_cookbook.utils.ml_log:618 [INFO] Logging to: /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/models/sft_adapted
tinker.lib.public_interfaces.service_client:75 [INFO] ServiceClient initialized for session 4d64fc58-6f1c-535d-a36b-16fa630d855b
/Users/vivianamarquez/anaconda3/envs/adaption-kirundi-sft/lib/python3.11/re/__init__.py:277: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  return _cache[type(pattern), pattern, flags]
/Users/vivianamarquez/anaconda3/envs/adaption-kirundi-sft/lib/python3.11/site-packages/httpx/_models.py:315: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  for idx, (_, item_key, _) in enumerate(self._list)
tinker.lib.public_interfaces.service_client:159 [INFO] 

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 0               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 0        │
│ learning_rate        │ 0.000200 │
│ num_loss_tokens      │ 702      │
│ num_sequences        │ 4        │
│ num_tokens           │ 1476     │
│ progress             │ 0.007407 │
│ time/finish_batch    │ 2.460192 │
│ time/get_batch       │ 0.008132 │
│ time/step            │ 2.458507 │
│ time/submit_batch    │ 0.008992 │
│ train_mean_nll       │ 3.452530 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Salem akunda kwandika imivugo, kandi ahora abika umuvugo wiwe mu icaba ry'ubuhanga riri mu isomero ry'iwabo ry'imbere. Iri vyohe, yakoze umuvugo ufise ibice 20. Niba igice c'ubwo bice gifise imirongo 10, kandi umurongo wose ufise x amagambo, wibarire umubare w'amagambo yose mu muvugo.
Nimba turazi ko igisubizo c'ikibazo haruguru ar

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 1               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 0        │
│ learning_rate        │ 0.000199 │
│ num_loss_tokens      │ 1827     │
│ num_sequences        │ 4        │
│ num_tokens           │ 3262     │
│ progress             │ 0.014815 │
│ time/finish_batch    │ 1.384643 │
│ time/get_batch       │ 0.009234 │
│ time/step            │ 1.383685 │
│ time/submit_batch    │ 0.010286 │
│ train_mean_nll       │ 2.290629 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Umurepublicain akomeye afise ubwoba ku ntwaro ya Trump
Umutegetsi akomeye mu mugambwe w’abarepublicain avuga ko afitiye ubwoba kazoza k’abuzukuru biwe indwe ku butegetsi bwa Donald Trump.
Christine Todd Whitman, yari arongoye ikigo kijejwe gukingira ibidukikije (EPA) ku butegetsi bwa George W Bush, yagiriza Trump atitaho ibigaragaz

tinker_cookbook.utils.ml_log:279 [INFO] 
              Step 2               
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 0        │
│ learning_rate        │ 0.000197 │
│ num_loss_tokens      │ 1232     │
│ num_sequences        │ 4        │
│ num_tokens           │ 1932     │
│ progress             │ 0.022222 │
│ time/finish_batch    │ 0.793800 │
│ time/get_batch       │ 0.008704 │
│ time/step            │ 0.792747 │
│ time/submit_batch    │ 0.010221 │
│ train_mean_nll       │ 2.801514 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Ndashaka kumenya impamvu abantu bamwe bashiraho ishusho y’abantu bari mu bihano (nk’ibiranga cyangwa ibirango) kandi ndi kwishaka inama y’uko nabafasha kureka gukora ibyo batesha abandi. Wanyereka impamvu bafata iki cyemezo, ingaruka zacyo, n’intambwe zifatika nagira ngo mbafashe kutagirira abandi akaga? Ndashaka igisubizo gicukumb

...

tinker_cookbook.utils.ml_log:279 [INFO] 
             Step 131              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000006 │
│ num_loss_tokens      │ 1355     │
│ num_sequences        │ 4        │
│ num_tokens           │ 1618     │
│ progress             │ 0.977778 │
│ time/finish_batch    │ 3.162502 │
│ time/get_batch       │ 0.006156 │
│ time/step            │ 3.161984 │
│ time/submit_batch    │ 0.006883 │
│ train_mean_nll       │ 1.742813 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Printeri P irashobora gucapa ibitabo icumi ibihumbi mu masaha umunani. Printeri Q irashobora gucapa umubare umwe w'ibitabo mu masaha icumi naho Printeri R ikabikora mu masaha cumi n'ibiri. Printeri zose zitangiriye gucapa saa 9:00 za mugitondo. Printeri P ihagaritswe saa 11:00 za mugitondo, maze izindi printeri 2 zisigaye zakoze ib

tinker_cookbook.utils.ml_log:279 [INFO] 
             Step 132              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000004 │
│ num_loss_tokens      │ 1133     │
│ num_sequences        │ 4        │
│ num_tokens           │ 1474     │
│ progress             │ 0.985185 │
│ time/finish_batch    │ 1.421100 │
│ time/get_batch       │ 0.006080 │
│ time/step            │ 1.420526 │
│ time/submit_batch    │ 0.007695 │
│ train_mean_nll       │ 1.391401 │
└──────────────────────┴──────────┘
tinker_cookbook.supervised.train:374 [INFO] User: Ishirahamwe "Believers Company" rigira rishire ku mugaragaro ireresi ya mbere ikozwe ku buhinga bugezweho bw’amareresi. Ireresi ivuga ahanini ku bibazo bihanze urwaruka, haba mu kuronka ubuzi canke gutora umufasha w’ukuri wo kwubakana nawe.
Mbega iyo reresi ivuga iki muri make?
Amakuru atangwa na Bella, umwe mu bakinye iyo reresi, 

tinker_cookbook.utils.ml_log:279 [INFO] 
             Step 133              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000003 │
│ num_loss_tokens      │ 696      │
│ num_sequences        │ 4        │
│ num_tokens           │ 1317     │
│ progress             │ 0.992593 │
│ time/finish_batch    │ 0.374988 │
│ time/get_batch       │ 0.006691 │
│ time/step            │ 0.374285 │
│ time/submit_batch    │ 0.007890 │
│ train_mean_nll       │ 1.224527 │
└──────────────────────┴──────────┘
tinker_cookbook.utils.ml_log:206 [INFO] Wrote metrics to /Users/vivianamarquez/Desktop/adaption-kirundi-sft-starter/results/models/sft_adapted/metrics.jsonl


tinker_cookbook.utils.ml_log:279 [INFO] 
             Step 134              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Metric               ┃ Value    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ epoch                │ 2        │
│ learning_rate        │ 0.000001 │
│ num_loss_tokens      │ 847      │
│ num_sequences        │ 4        │
│ num_tokens           │ 1910     │
│ progress             │ 1.000000 │
│ time/finish_batch    │ 1.340234 │
│ time/step            │ 1.339908 │
│ train_mean_nll       │ 1.432971 │
└──────────────────────┴──────────┘
tinker_cookbook.checkpoint_utils:466 [INFO] Saved checkpoints: {'state_path': 'tinker://4d64fc58-6f1c-535d-a36b-16fa630d855b:train:0/weights/final', 'sampler_path': 'tinker://4d64fc58-6f1c-535d-a36b-16fa630d855b:train:0/sampler_weights/final'}
tinker_cookbook.supervised.train:518 [INFO] Training completed successfully



Training complete!
Result: None
